In [11]:
import PhidgetUtils
import numpy as np
import tkinter as tk
from tkinter import ttk
import uuid
from math import ceil
import os

from time import sleep, time

FREQUENCY = 10

In [12]:
acc = PhidgetUtils.PhidgetAccelerometer(data_rate=FREQUENCY)

while not acc.is_attached():
    sleep(0.1)

print("Accelerometer is attached!")

PhidgetException: PhidgetException 0x03 (Timed Out)
No Phidgets were detected at all. Make sure your device is attached.

In [ ]:
bar = PhidgetUtils.PhidgetBarometer(data_rate=FREQUENCY)

while not bar.is_attached():
    sleep(0.1)

print("Barometer is attached!")

Barometer is attached!


In [4]:
def magnitude(v):
    return v[0] * v[0] + v[1] * v[1] + v[2] * v[2]

def sign(v):
    return 1 if v > 0 else -1

In [ ]:
RECORDING = False

folder = f"./datastreams/{time()}"
os.makedirs(folder, exist_ok=True)

dtype = [('timestamp', 'int32'), ('acceleration', 'float32'), ('pressure', 'float32')]

data_size_from_mins = lambda mins : int(FREQUENCY * 60 * mins)

data_size = data_size_from_mins(15)
data = np.memmap(f"{folder}/0.dat", dtype=dtype, mode='w+', shape=(data_size,))
data_index = 0
file_count = 1

prev_time = -1

root = tk.Tk()
root.title("Accelerometer + Barometer Measurements")
root.geometry("400x250")
root.resizable(False, False)

''

In [6]:
start_button = stop_button = None

def start():
    global RECORDING
    RECORDING = True
    stop_button.config(state=tk.NORMAL)
    start_button.config(state=tk.DISABLED)
    record()

def stop():
    global RECORDING
    RECORDING = False
    stop_button.config(state=tk.DISABLED)
    start_button.config(state=tk.NORMAL)

def record():

    global prev_time
    global data_index
    global data
    global file_count
    global folder
    global data_size

    if RECORDING:
        if acc.getTimestamp() > prev_time:
            prev_time = acc.getTimestamp()

            if data_size <= data_index:
                data = np.memmap(f"{folder}/{file_count}.dat", dtype=dtype, mode='w+', shape=(data_size,))
                file_count += 1
                data_index = 0

            data[data_index] = (
                acc.getTimestamp(),
                magnitude(acc.getAcceleration()) * sign(acc.getAcceleration()[2]),
                bar.getPressure()
            )
            data_index += 1
            
        root.after(ceil(1000 / FREQUENCY), record)

In [7]:
# Styling with ttk
style = ttk.Style()
style.configure("TButton", font=("Arial", 12), padding=10)
style.configure("TLabel", font=("Arial", 14))
style.configure("TFrame", background="#f0f0f0")

# Main frame
frame = ttk.Frame(root, padding=20, style="TFrame")
frame.pack(expand=True, fill=tk.BOTH)

# Title Label
title_label = ttk.Label(
    frame, text="Accelerometer + Barometer", font=("Arial", 16, "bold")
)
title_label.pack(pady=10)

# Status Label
status_label = ttk.Label(frame, text="Ready to Start", foreground="blue")
status_label.pack(pady=10)

# Buttons
button_frame = ttk.Frame(frame, padding=10, style="TFrame")
button_frame.pack(pady=20)

start_button = ttk.Button(button_frame, text="Start", command=start)
start_button.grid(row=0, column=0, padx=10)

stop_button = ttk.Button(button_frame, text="Stop", command=stop, state=tk.DISABLED)
stop_button.grid(row=0, column=1, padx=10)

# Run the application
root.mainloop()


In [8]:
acc.stop()
bar.stop()

In [10]:
import numpy as np
import os

# Path to the memory-mapped file
file_path = './datastreams/1736249834.1298068/0.dat'

dtype = [('timestamp', 'int32'), ('acceleration', 'float32'), ('pressure', 'float32')]

# Open the memory-mapped file in read-only mode
data = np.memmap(file_path, dtype=dtype, mode='r')

# Access elements
print(data[:20])

[(7608, 0.03277018, 98.419) (7712, 0.03233289, 98.402)
 (7816, 0.03330282, 98.399) (7920, 0.03305615, 98.408)
 (8024, 0.03134472, 98.408) (8128, 0.0323717 , 98.403)
 (8336, 0.03301542, 98.398) (8440, 0.03271422, 98.391)
 (8648, 0.03217271, 98.388) (8752, 0.0324866 , 98.409)
 (8856, 0.03377605, 98.412) (8960, 0.03317045, 98.394)
 (9064, 0.03182873, 98.406) (9168, 0.03332997, 98.404)
 (9272, 0.03275967, 98.405) (9376, 0.03282642, 98.406)
 (9480, 0.03289618, 98.404) (9584, 0.03331333, 98.405)
 (9688, 0.0326976 , 98.4  ) (9792, 0.031981  , 98.408)]
